In [1]:
import sys
import requests
import pandas as pd
import folium

print("Versão do Python:", sys.version)
print("Executável:", sys.executable)

print("\nBibliotecas carregadas:")
print("Requests:", requests.__version__)
print("Pandas:", pd.__version__)
print("Folium:", folium.__version__)

Versão do Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Executável: c:\Users\gsram\OneDrive\Documentos\ADS\Atividades\Algoritimos\.venv\Scripts\python.exe

Bibliotecas carregadas:
Requests: 2.34.2
Pandas: 3.1.1
Folium: 1.0.0


In [3]:
from getpass import getpass
import requests

# Endereço oficial da API Olho Vivo
BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1"

# Solicita o token sem exibi-lo na tela
token = getpass("Digite seu token da SPTrans: ")

# Criamos uma sessão para manter a autenticação
sessao_api = requests.Session()

try:
    resposta = sessao_api.post(
        f"{BASE_URL}/Login/Autenticar",
        params={"token": token},
        timeout=20
    )

    if resposta.status_code == 200:

        autenticado = resposta.json()

        if autenticado is True:
            print("Autenticação realizada com sucesso!")
            print("Conexão com a API Olho Vivo estabelecida.")

        else:
            print("A SPTrans não confirmou a autenticação.")

    else:
        print("Falha na autenticação.")
        print("Código HTTP:", resposta.status_code)

except requests.exceptions.Timeout:
    print("Tempo limite excedido na comunicação com a SPTrans.")

except requests.exceptions.ConnectionError:
    print("Não foi possível conectar à API.")

except requests.exceptions.RequestException:
    print("Ocorreu um erro durante a requisição.")

finally:
    token = None

Autenticação realizada com sucesso!
Conexão com a API Olho Vivo estabelecida.


In [4]:
resposta = sessao_api.get(
    f"{BASE_URL}/Linha/Buscar",
    params={"termosBusca": "8000"},
    timeout=20
)

resposta.raise_for_status()

linhas = resposta.json()

df_linhas = pd.DataFrame(linhas)

print("Quantidade de registros encontrados:", len(df_linhas))

display(df_linhas)

Quantidade de registros encontrados: 4


,cl,lc,lt,sl,tl,tp,ts
0,2506,False,8000,1,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
1,35274,False,8000,2,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
2,1273,False,8000,1,10,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
3,34041,False,8000,2,10,PÇA. RAMOS DE AZEVEDO,TERM. LAPA


In [5]:
# Códigos das duas direções da linha 8000
linhas_teste = {
    "8000 - Sentido 1": 1273,
    "8000 - Sentido 2": 34041
}

# Armazenar os resultados
paradas_por_linha = {}

for nome_linha, codigo in linhas_teste.items():

    resposta = sessao_api.get(
        f"{BASE_URL}/Parada/BuscarParadasPorLinha",
        params={"codigoLinha": codigo},
        timeout=20
    )

    resposta.raise_for_status()

    dados = resposta.json()

    paradas_por_linha[codigo] = dados

    print(f"\n{nome_linha}")
    print(f"Código identificador: {codigo}")
    print(f"Quantidade de paradas: {len(dados)}")

    df_paradas = pd.DataFrame(dados)

    display(df_paradas.head(10))


8000 - Sentido 1
Código identificador: 1273
Quantidade de paradas: 14


,cp,np,ed,py,px
0,7014417,ANGELICA B/C,AV ANGELICA,-23.534564,-46.654302
1,60016784,PARADA PALMEIRAS B/C,R PADRE ANTONIO TOMAS/ AV POMPEIA,-23.525799,-46.679251
2,60016786,ANTARTICA B/C,PC SOUSA ARANHA/ VDTO ANTARTICA,-23.526523,-46.673588
3,60016788,BABY BARIONY B/C,R DONA GERMAINE BUCHARD/ R MINISTRO GODOI,-23.528131,-46.670652
4,60016790,PQ. DA ÁGUA BRANCA B/C,R MINISTRO GODOI/ R LINCOLN ALBUQUERQUE,-23.530210,-46.666900
5,480014612,SCIPIAO B/C,R SCIPIAO/ R AURELIA,-23.524940,-46.698995
6,480014613,VESPASIANO B/C,R VESPASIANO/ R CAIO GRACO,-23.524994,-46.695297
7,480014614,CORNELIA,R CLAUDIO/ R CRASSO,-23.525029,-46.690074
8,480014961,CATAO B/C,R FAUSTOLO/ R ROMA,-23.523407,-46.700541
9,610014603,SESC POMPEIA B/C,R TURIASSU/ R BARAO DO BANANAL,-23.525170,-46.683408



8000 - Sentido 2
Código identificador: 34041
Quantidade de paradas: 13


,cp,np,ed,py,px
0,614432,BABY BARIONY C/B,R PEDRO MACHADO,-23.527606,-46.671514
1,7014416,ANGELICA C/B,R DR. ALBUQUERQUE LINS,-23.534377,-46.654322
2,60014604,SESC POMPEIA C/B,VDTO POMPEIA/ R JOAQUIM FERREIRA,-23.525262,-46.682254
3,60016783,PARADA PALMEIRAS C/B,R PADRE ANTONIO TOMAS/ AV POMPEIA,-23.525653,-46.679493
4,60016785,ANTARTICA C/B,R ENGENHEIRO STEVENSON/ PC SOUSA ARANHA,-23.526220,-46.674187
5,60016789,PQ. DA ÁGUA BRANCA C/B,R DONA GERMAINE BUCHARD/ R MINISTRO GODOI,-23.529636,-46.667777
6,480014608,TIBERIO C/B,R TIBERIO/ R MENFIS,-23.522875,-46.688219
7,480014609,DUILIO C/B,R DUILIO/ R CLAUDIO,-23.522627,-46.692121
8,480014610,TENDAL DA LAPA C/B,R ESPARTACO/ R MORAIS E SILVA,-23.522339,-46.696246
9,700006085,ROSA E SILVA C/B,R OLIMPIA DE ALMEIDA PRADO/ R LOPES DE OLIVEIRA,-23.532629,-46.657465


In [6]:
# Criar DataFrames para cada sentido
df_paradas_s1 = pd.DataFrame(
    paradas_por_linha[1273]
).copy()

df_paradas_s2 = pd.DataFrame(
    paradas_por_linha[34041]
).copy()

# Validar dados de cada sentido
def validar_paradas(df, nome):

    # Converter coordenadas para números
    df["py"] = pd.to_numeric(df["py"], errors="coerce")
    df["px"] = pd.to_numeric(df["px"], errors="coerce")

    # Verificar coordenadas geograficamente válidas
    coordenadas_validas = (
        df["py"].between(-90, 90) &
        df["px"].between(-180, 180)
    )

    print(f"\n--- {nome} ---")

    print("Total de paradas:", len(df))
    print("Coordenadas ausentes:",
          df[["py", "px"]].isna().any(axis=1).sum())

    print("Coordenadas válidas:",
          coordenadas_validas.sum())

    print("Códigos de parada duplicados:",
          df["cp"].duplicated().sum())

    return df

# Executar validação
df_paradas_s1 = validar_paradas(
    df_paradas_s1, "Sentido 1"
)

df_paradas_s2 = validar_paradas(
    df_paradas_s2, "Sentido 2"
)


--- Sentido 1 ---
Total de paradas: 14
Coordenadas ausentes: 0
Coordenadas válidas: 14
Códigos de parada duplicados: 0

--- Sentido 2 ---
Total de paradas: 13
Coordenadas ausentes: 0
Coordenadas válidas: 13
Códigos de parada duplicados: 0


In [8]:
# Dados da consulta
codigo_linha_teste = 1273
codigo_parada_teste = 7014417

try:
    resposta = sessao_api.get(
        f"{BASE_URL}/Previsao",
        params={
            "codigoLinha": codigo_linha_teste,
            "codigoParada": codigo_parada_teste
        },
        timeout=20
    )

    resposta.raise_for_status()

    dados_previsao = resposta.json()

    # Extrair informações da parada
    parada = dados_previsao.get("p", {})

    # Extrair linhas e veículos
    linhas = parada.get("l", [])

    print("Horário de referência:",
          dados_previsao.get("hr"))

    print("Parada:",
          parada.get("np"))

    print("Código:",
          parada.get("cp"))

    # Identificar a linha consultada
    linha_encontrada = [
        linha for linha in linhas
        if str(linha.get("cl")) == str(codigo_linha_teste)
    ]

    if linha_encontrada:

        veiculos = []

        for linha in linha_encontrada:
            for veiculo in linha.get("vs", []):

                veiculos.append({
                    "Prefixo": veiculo.get("p"),
                    "Previsão": veiculo.get("t"),
                    "Acessível para PcD": veiculo.get("a"),
                    "Última atualização UTC": veiculo.get("ta")
                })

        if veiculos:
            df_previsoes = pd.DataFrame(veiculos)

            print("\nVeículos encontrados:")
            display(df_previsoes)

        else:
            print("\nLinha encontrada, mas não existem "
                  "veículos com previsão disponível no momento.")

    else:
        print("\nNão foi retornada previsão para "
              "essa linha e parada no momento.")

        print("Códigos de linhas retornados:",
              [linha.get("cl") for linha in linhas])

except requests.exceptions.Timeout:
    print("Tempo limite excedido.")

except requests.exceptions.HTTPError as erro:
    print("Erro HTTP:", erro)

except requests.exceptions.RequestException as erro:
    print("Erro na comunicação:", erro)

except ValueError:
    print("Não foi possível interpretar a resposta da API.")

AttributeError: 'NoneType' object has no attribute 'get'

In [9]:
import json

codigo_linha_teste = 1273
codigo_parada_teste = 7014417

try:
    resposta_previsao = sessao_api.get(
        f"{BASE_URL}/Previsao",
        params={
            "codigoLinha": codigo_linha_teste,
            "codigoParada": codigo_parada_teste
        },
        timeout=20
    )

    print("Status HTTP:", resposta_previsao.status_code)
    print("Content-Type:",
          resposta_previsao.headers.get("Content-Type"))

    resposta_previsao.raise_for_status()

    dados_previsao = resposta_previsao.json()

    print("\nTipo da resposta:",
          type(dados_previsao).__name__)

    print("\nConteúdo retornado pela SPTrans:")

    print(json.dumps(
        dados_previsao,
        indent=4,
        ensure_ascii=False
    ))

except requests.exceptions.RequestException as erro:
    print("Erro na comunicação:", erro)

except ValueError:
    print("A resposta não contém um JSON válido.")
    print(resposta_previsao.text)

Status HTTP: 200
Content-Type: application/json; charset=utf-8

Tipo da resposta: dict

Conteúdo retornado pela SPTrans:
{
    "hr": "09:59",
    "p": {
        "cp": 7014417,
        "np": "ANGELICA B/C",
        "py": -23.534564,
        "px": -46.654302,
        "l": [
            {
                "c": "8000-10",
                "cl": 1273,
                "sl": 1,
                "lt0": "PÇA. RAMOS DE AZEVEDO",
                "lt1": "TERM. LAPA",
                "qv": 1,
                "vs": [
                    {
                        "p": "11525",
                        "t": "10:05",
                        "a": true,
                        "ta": "2026-11-19T12:58:56Z",
                        "py": -23.527763333333336,
                        "px": -46.67136000000001,
                        "sv": null,
                        "is": null
                    }
                ]
            }
        ]
    }
}


In [10]:
from datetime import datetime

# Utiliza a resposta já recebida no teste anterior

if not isinstance(dados_previsao, dict):

    print("Resposta inválida da API.")

else:

    parada = dados_previsao.get("p")

    if not isinstance(parada, dict):

        print("Informações da parada indisponíveis.")

    else:

        horario_referencia = dados_previsao.get("hr")

        print("Parada:", parada.get("np"))
        print("Horário de referência:", horario_referencia)

        linhas_retorno = parada.get("l") or []

        registros = []

        for linha in linhas_retorno:

            if not isinstance(linha, dict):
                continue

            veiculos = linha.get("vs") or []

            for veiculo in veiculos:

                if not isinstance(veiculo, dict):
                    continue

                horario_previsto = veiculo.get("t")

                # Calcular minutos de espera, quando possível
                minutos = None

                try:
                    referencia = datetime.strptime(
                        horario_referencia, "%H:%M"
                    )

                    previsto = datetime.strptime(
                        horario_previsto, "%H:%M"
                    )

                    diferenca = int(
                        (previsto - referencia).total_seconds() / 60
                    )

                    if diferenca < 0:
                        diferenca += 1440

                    minutos = diferenca

                except (ValueError, TypeError):
                    pass

                registros.append({
                    "Linha": linha.get("c"),
                    "Sentido": linha.get("sl"),
                    "Prefixo": veiculo.get("p"),
                    "Previsão": horario_previsto,
                    "Espera aproximada (min)": minutos,
                    "Acessível": veiculo.get("a")
                })

        if registros:

            df_previsoes = pd.DataFrame(registros)

            print("\nPrevisões disponíveis:")
            display(df_previsoes)

        else:

            print("\nNenhum veículo com previsão disponível.")

Parada: ANGELICA B/C
Horário de referência: 09:59

Previsões disponíveis:


,Linha,Sentido,Prefixo,Previsão,Espera aproximada (min),Acessível
0,8000-10,1,11525,10:05,6,True


In [11]:
# Consultar previsões em todas as paradas da linha
codigo_linha = 1273

resposta = sessao_api.get(
    f"{BASE_URL}/Previsao/Linha",
    params={"codigoLinha": codigo_linha},
    timeout=30
)

resposta.raise_for_status()

dados_linha = resposta.json()

print("Horário de referência:", dados_linha.get("hr"))

paradas_previsao = dados_linha.get("ps") or []

registros = []

for parada in paradas_previsao:

    for veiculo in parada.get("vs") or []:

        registros.append({
            "Codigo_parada": parada.get("cp"),
            "Nome_parada": parada.get("np"),
            "Prefixo_veiculo": veiculo.get("p"),
            "Previsao": veiculo.get("t"),
            "Acessivel": veiculo.get("a")
        })

df_previsoes_linha = pd.DataFrame(registros)

print("\nQuantidade de paradas com previsão:",
      len(paradas_previsao))

print("Quantidade de previsões encontradas:",
      len(df_previsoes_linha))

display(df_previsoes_linha)

Horário de referência: 10:07

Quantidade de paradas com previsão: 14
Quantidade de previsões encontradas: 2


,Codigo_parada,Nome_parada,Prefixo_veiculo,Previsao,Acessivel
0,700016623,ANA CINTRA B/C,11525,10:10,True
1,670010530,VITORIA B/C,11525,10:12,True


In [12]:
# Consultar localização dos veículos da linha 8000 - Sentido 1

resposta = sessao_api.get(
    f"{BASE_URL}/Posicao/Linha",
    params={"codigoLinha": 1273},
    timeout=20
)

resposta.raise_for_status()

dados_posicao = resposta.json()

veiculos = dados_posicao.get("vs") or []

print("Horário de referência:", dados_posicao.get("hr"))
print("Quantidade de veículos:", len(veiculos))

df_veiculos = pd.DataFrame([
    {
        "Prefixo": v.get("p"),
        "Acessível": v.get("a"),
        "Última atualização": v.get("ta"),
        "Latitude": v.get("py"),
        "Longitude": v.get("px")
    }
    for v in veiculos
])

display(df_veiculos)

Horário de referência: 10:16
Quantidade de veículos: 2


,Prefixo,Acessível,Última atualização,Latitude,Longitude
0,11524,True,2026-11-19T13:16:37Z,-23.524963,-46.694177
1,11525,True,2026-11-19T13:16:44Z,-23.544507,-46.644027


In [13]:
# Consultar posições de todos os veículos monitorados

resposta = sessao_api.get(
    f"{BASE_URL}/Posicao",
    timeout=60
)

resposta.raise_for_status()

dados_posicao_geral = resposta.json()

linhas_monitoradas = dados_posicao_geral.get("l") or []

# Organizar informações das linhas
registros = []

for linha in linhas_monitoradas:

    veiculos = linha.get("vs") or []

    registros.append({
        "Codigo_linha": linha.get("cl"),
        "Letreiro": linha.get("c"),
        "Sentido": linha.get("sl"),
        "Destino": linha.get("lt0"),
        "Origem": linha.get("lt1"),
        "Veiculos_localizados": len(veiculos)
    })

df_linhas_monitoradas = pd.DataFrame(registros)

print("Horário de referência:",
      dados_posicao_geral.get("hr"))

print("Registros de linhas:",
      len(df_linhas_monitoradas))

print("Identificadores únicos:",
      df_linhas_monitoradas["Codigo_linha"].nunique())

print("Total de veículos:",
      df_linhas_monitoradas["Veiculos_localizados"].sum())

print("\nPrimeiras 15 linhas:")

display(df_linhas_monitoradas.head(15))

AttributeError: 'NoneType' object has no attribute 'get'

In [14]:
# Diagnóstico da consulta geral de posições

resposta_diagnostico = sessao_api.get(
    f"{BASE_URL}/Posicao",
    timeout=60
)

print("Status HTTP:", resposta_diagnostico.status_code)

print("URL consultada:", resposta_diagnostico.url)

print("Content-Type:",
      resposta_diagnostico.headers.get("Content-Type"))

print("Tamanho da resposta:",
      len(resposta_diagnostico.content), "bytes")

print("\nConteúdo original da resposta:")

print(repr(resposta_diagnostico.text[:3000]))

try:
    dados_diagnostico = resposta_diagnostico.json()

    print("\nTipo de dado recebido:",
          type(dados_diagnostico).__name__)

    if isinstance(dados_diagnostico, dict):
        print("Campos encontrados:",
              list(dados_diagnostico.keys()))

        print("Horário de referência:",
              dados_diagnostico.get("hr"))

        print("Quantidade de linhas:",
              len(dados_diagnostico.get("l") or []))

    elif dados_diagnostico is None:
        print("\nA API retornou JSON null.")

    else:
        print("\nA estrutura retornada é diferente da esperada.")

except ValueError:
    print("\nA resposta não contém JSON válido.")

Status HTTP: 200
URL consultada: https://api.olhovivo.sptrans.com.br/v2.1/Posicao
Content-Type: application/json; charset=utf-8
Tamanho da resposta: 1329507 bytes

Conteúdo original da resposta:
'{"hr":"10:39","l":[{"c":"702U-10","cl":34098,"sl":2,"lt0":"TERM. PQ. D. PEDRO II","lt1":"CID. UNIVERSITÁRIA","qv":8,"vs":[{"p":82454,"a":true,"ta":"2026-11-19T13:39:03Z","py":-23.559549500000003,"px":-46.736019,"sv":null,"is":null},{"p":82770,"a":true,"ta":"2026-11-19T13:38:49Z","py":-23.5643695,"px":-46.678257,"sv":null,"is":null},{"p":82553,"a":true,"ta":"2026-11-19T13:39:17Z","py":-23.5521175,"px":-46.7315415,"sv":null,"is":null},{"p":82453,"a":true,"ta":"2026-11-19T13:39:18Z","py":-23.54805,"px":-46.629795,"sv":null,"is":null},{"p":82552,"a":true,"ta":"2026-11-19T13:39:17Z","py":-23.5521175,"px":-46.7315415,"sv":null,"is":null},{"p":82771,"a":true,"ta":"2026-11-19T13:39:17Z","py":-23.5521175,"px":-46.7315415,"sv":null,"is":null},{"p":82763,"a":true,"ta":"2026-11-19T13:39:18Z","py":-23.5479

In [15]:
# Recuperar a resposta válida do diagnóstico anterior
dados_posicao_geral = dados_diagnostico

if not isinstance(dados_posicao_geral, dict):
    raise ValueError("A resposta da API não contém um objeto válido.")

linhas_monitoradas = dados_posicao_geral.get("l") or []

registros_linhas = []
registros_veiculos = []

for linha in linhas_monitoradas:

    veiculos = linha.get("vs") or []

    registros_linhas.append({
        "Codigo_linha": linha.get("cl"),
        "Letreiro": linha.get("c"),
        "Sentido": linha.get("sl"),
        "Destino": linha.get("lt0"),
        "Origem": linha.get("lt1"),
        "Quantidade_veiculos": linha.get("qv"),
        "Veiculos_recebidos": len(veiculos)
    })

    for veiculo in veiculos:

        registros_veiculos.append({
            "Codigo_linha": linha.get("cl"),
            "Prefixo": veiculo.get("p"),
            "Latitude": veiculo.get("py"),
            "Longitude": veiculo.get("px"),
            "Ultima_atualizacao": veiculo.get("ta")
        })

# Converter para DataFrames
df_linhas_monitoradas = pd.DataFrame(registros_linhas)
df_veiculos_monitorados = pd.DataFrame(registros_veiculos)

# Apresentar estatísticas
print("HORÁRIO DA CONSULTA:",
      dados_posicao_geral.get("hr"))

print("\nESTATÍSTICAS DAS LINHAS")

print("Registros retornados:",
      len(df_linhas_monitoradas))

print("Códigos de linha únicos:",
      df_linhas_monitoradas["Codigo_linha"].nunique())

print("Registros com código duplicado:",
      df_linhas_monitoradas["Codigo_linha"].duplicated().sum())

print("\nVEÍCULOS")

print("Total de posições recebidas:",
      len(df_veiculos_monitorados))

print("Total informado pelo campo qv:",
      pd.to_numeric(
          df_linhas_monitoradas["Quantidade_veiculos"],
          errors="coerce"
      ).sum())

print("\nDISTRIBUIÇÃO POR SENTIDO")

display(
    df_linhas_monitoradas["Sentido"]
    .value_counts()
    .sort_index()
    .rename_axis("Sentido")
    .to_frame("Quantidade de registros")
)

print("\nEXEMPLO: LINHA 8000")

display(
    df_linhas_monitoradas[
        df_linhas_monitoradas["Letreiro"].astype(str) == "8000"
    ]
)

print("\nPRIMEIRAS 10 LINHAS MONITORADAS")

display(df_linhas_monitoradas.head(10))

HORÁRIO DA CONSULTA: 10:39

ESTATÍSTICAS DAS LINHAS
Registros retornados: 2060
Códigos de linha únicos: 2060
Registros com código duplicado: 0

VEÍCULOS
Total de posições recebidas: 10598
Total informado pelo campo qv: 10598

DISTRIBUIÇÃO POR SENTIDO


,Quantidade de registros
Sentido,
1,1026
2,1034



EXEMPLO: LINHA 8000


,Codigo_linha,Letreiro,Sentido,Destino,Origem,Quantidade_veiculos,Veiculos_recebidos



PRIMEIRAS 10 LINHAS MONITORADAS


,Codigo_linha,Letreiro,Sentido,Destino,Origem,Quantidade_veiculos,Veiculos_recebidos
0,34098,702U-10,2,TERM. PQ. D. PEDRO II,CID. UNIVERSITÁRIA,8,8
1,33819,3761-10,2,METRÔ CARRÃO,3A. DIVISÃO,8,8
2,1851,5109-10,1,TERM. MERCADO,TERM. VL. PRUDENTE,1,1
3,90,6027-10,1,TERM. STO. AMARO,JD. GRAUNA,6,6
4,33831,3785-10,2,METRÔ ITAQUERA,COHAB BARRO BRANCO,6,6
5,1926,857P-10,1,PARAÍSO,TERM. CAMPO LIMPO,8,8
6,33342,8215-10,2,PÇA. DO CORREIO,JD. PAULISTANO,5,5
7,1042,3742-10,1,METRÔ ITAQUERA,JD. LARANJEIRA,8,8
8,34967,407L-10,2,METRÔ GUILHERMINA/ESPERANÇA,BARRO BRANCO,16,16
9,32991,273G-10,2,METRÔ ARTUR ALVIM,JD. HELENA,9,9


In [16]:
# Teste de pesquisa abrangente de linhas

try:

    resposta_busca = sessao_api.get(
        f"{BASE_URL}/Linha/Buscar",
        params={"termosBusca": ""},
        timeout=30
    )

    print("Status HTTP:", resposta_busca.status_code)

    resposta_busca.raise_for_status()

    dados_busca = resposta_busca.json()

    print("\nTipo da resposta:",
          type(dados_busca).__name__)

    if isinstance(dados_busca, list):

        df_catalogo_teste = pd.DataFrame(dados_busca)

        print("Quantidade de registros:",
              len(df_catalogo_teste))

        if not df_catalogo_teste.empty:

            coluna_codigo = (
                "cl" if "cl" in df_catalogo_teste.columns
                else "CodigoLinha"
            )

            print("Identificadores únicos:",
                  df_catalogo_teste[coluna_codigo].nunique())

            display(df_catalogo_teste.head(15))

        else:

            print("A API retornou uma lista vazia.")

    else:

        print("Estrutura diferente da esperada:")
        print(dados_busca)

except requests.exceptions.RequestException as erro:

    print("Erro na consulta:", erro)

except ValueError:

    print("A API não retornou um JSON válido.")

Status HTTP: 200

Tipo da resposta: list
Quantidade de registros: 0
A API retornou uma lista vazia.


In [17]:
# Procurar a linha 8000 pelo início do letreiro

resultado_8000 = df_linhas_monitoradas[
    df_linhas_monitoradas["Letreiro"]
    .astype(str)
    .str.startswith("8000")
]

print("Registros da linha 8000 encontrados:",
      len(resultado_8000))

display(resultado_8000)

Registros da linha 8000 encontrados: 4


,Codigo_linha,Letreiro,Sentido,Destino,Origem,Quantidade_veiculos,Veiculos_recebidos
663,34041,8000-10,2,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,3,3
791,1273,8000-10,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,2,2
876,2506,8000-1,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,2,2
1942,35274,8000-1,2,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,4,4


In [18]:
import math

# Coordenadas de teste
origem_teste = (-23.534564, -46.654302)
destino_teste = (-23.525799, -46.679251)

# Raio experimental para encontrar linhas candidatas
raio_pre_selecao = 3000

# Função de distância Haversine
def haversine(lat1, lon1, lat2, lon2):

    raio_terra = 6371000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return 2 * raio_terra * math.asin(math.sqrt(a))


# Copiar os veículos monitorados
df_teste = df_veiculos_monitorados.copy()

# Garantir que as coordenadas sejam numéricas
for coluna in ["Latitude", "Longitude"]:
    df_teste[coluna] = pd.to_numeric(
        df_teste[coluna], errors="coerce"
    )

df_teste = df_teste.dropna(
    subset=["Codigo_linha", "Latitude", "Longitude"]
)

# Calcular distância dos veículos até a origem
df_teste["Distancia_origem"] = df_teste.apply(
    lambda v: haversine(
        v["Latitude"],
        v["Longitude"],
        *origem_teste
    ),
    axis=1
)

# Calcular distância dos veículos até o destino
df_teste["Distancia_destino"] = df_teste.apply(
    lambda v: haversine(
        v["Latitude"],
        v["Longitude"],
        *destino_teste
    ),
    axis=1
)

# Resumir as menores distâncias por linha
resumo = df_teste.groupby("Codigo_linha").agg(
    Veiculo_mais_proximo_origem=(
        "Distancia_origem", "min"
    ),
    Veiculo_mais_proximo_destino=(
        "Distancia_destino", "min"
    )
).reset_index()

# Identificar linhas próximas de pelo menos um ponto
candidatas = resumo[
    (resumo["Veiculo_mais_proximo_origem"]
     <= raio_pre_selecao)
    |
    (resumo["Veiculo_mais_proximo_destino"]
     <= raio_pre_selecao)
]

# Relacionar com os dados cadastrais da consulta
candidatas = candidatas.merge(
    df_linhas_monitoradas,
    on="Codigo_linha",
    how="left"
)

candidatas = candidatas.sort_values(
    by=[
        "Veiculo_mais_proximo_origem",
        "Veiculo_mais_proximo_destino"
    ]
)

print("LINHAS PRÓXIMAS DA ORIGEM:")
print(
    (
        resumo["Veiculo_mais_proximo_origem"]
        <= raio_pre_selecao
    ).sum()
)

print("\nLINHAS PRÓXIMAS DO DESTINO:")
print(
    (
        resumo["Veiculo_mais_proximo_destino"]
        <= raio_pre_selecao
    ).sum()
)

print("\nTOTAL DE LINHAS CANDIDATAS:")
print(len(candidatas))

print("\nLINHAS RELACIONADAS À 8000:")
display(
    candidatas[
        candidatas["Letreiro"]
        .astype(str)
        .str.startswith("8000")
    ]
)

print("\nPRIMEIRAS 20 CANDIDATAS:")
display(candidatas.head(20))

LINHAS PRÓXIMAS DA ORIGEM:
314

LINHAS PRÓXIMAS DO DESTINO:
194

TOTAL DE LINHAS CANDIDATAS:
395

LINHAS RELACIONADAS À 8000:


,Codigo_linha,Veiculo_mais_proximo_origem,Veiculo_mais_proximo_destino,Letreiro,Sentido,Destino,Origem,Quantidade_veiculos,Veiculos_recebidos
119,1273,765.915759,1957.945658,8000-10,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,2,2
321,34041,1110.885459,1614.093738,8000-10,2,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,3,3
182,2506,1813.208646,2178.325397,8000-1,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,2,2
390,35274,4917.877834,2202.017655,8000-1,2,PÇA. RAMOS DE AZEVEDO,TERM. LAPA,4,4



PRIMEIRAS 20 CANDIDATAS:


,Codigo_linha,Veiculo_mais_proximo_origem,Veiculo_mais_proximo_destino,Letreiro,Sentido,Destino,Origem,Quantidade_veiculos,Veiculos_recebidos
259,33342,118.672738,2598.675459,8215-10,2,PÇA. DO CORREIO,JD. PAULISTANO,5,5
342,34149,285.128991,2830.415952,877T-10,2,METRÔ PARAÍSO,VL. ANASTÁCIO,5,5
68,574,331.064815,2607.574264,8215-10,1,PÇA. DO CORREIO,JD. PAULISTANO,4,4
197,32892,376.909722,2849.500476,805L-10,2,ACLIMAÇÃO,TERM. PRINC. ISABEL,4,4
262,33348,394.186587,1970.233532,8549-10,2,PÇA. DO CORREIO,TAIPAS,8,8
232,33211,394.186587,2460.925593,8677-10,2,LGO. DO PAISSANDU,JD. LÍBANO,8,8
65,560,409.674335,2665.089762,178L-10,1,HOSP. DAS CLÍNICAS,LAUZANE PAULISTA,10,10
273,33398,443.729449,2561.585920,177H-10,2,CID. UNIVERSITÁRIA,METRÔ SANTANA,8,8
48,469,476.982314,1448.261892,719R-10,1,METRÔ BARRA FUNDA,RIO PEQUENO,7,7
278,33439,605.291482,3096.067031,1732-10,2,TERM. AMARAL GURGEL,VL. SABRINA,8,8


In [19]:
# Variantes da linha 8000 identificadas anteriormente
codigos_linha_8000 = [1273, 34041, 2506, 35274]

# Metadados das linhas monitoradas
metadados_linhas = (
    df_linhas_monitoradas
    [["Codigo_linha", "Letreiro", "Sentido"]]
    .drop_duplicates("Codigo_linha")
)

# Armazenar os resultados
resultados_paradas = []

# Raio definitivo de proximidade
raio_paradas = 500

for codigo in codigos_linha_8000:

    # Reutilizar dados já consultados
    if codigo in paradas_por_linha:

        dados = paradas_por_linha[codigo]

    else:

        # Consultar somente quando ainda não houver dados
        try:
            resposta = sessao_api.get(
                f"{BASE_URL}/Parada/BuscarParadasPorLinha",
                params={"codigoLinha": codigo},
                timeout=30
            )

            resposta.raise_for_status()
            dados = resposta.json()

            if not isinstance(dados, list):
                print(f"Resposta inválida para a linha {codigo}")
                continue

            paradas_por_linha[codigo] = dados

        except requests.exceptions.RequestException as erro:

            print(f"Erro na linha {codigo}: {erro}")
            continue

    if not dados:
        print(f"Nenhuma parada encontrada para {codigo}")
        continue

    df_temp = pd.DataFrame(dados)

    # Converter coordenadas
    df_temp["py"] = pd.to_numeric(
        df_temp["py"], errors="coerce"
    )

    df_temp["px"] = pd.to_numeric(
        df_temp["px"], errors="coerce"
    )

    df_temp = df_temp.dropna(subset=["py", "px"])

    if df_temp.empty:
        continue

    # Calcular distância de cada parada até a origem
    df_temp["Distancia_origem"] = df_temp.apply(
        lambda p: haversine(
            origem_teste[0],
            origem_teste[1],
            p["py"],
            p["px"]
        ),
        axis=1
    )

    # Calcular distância de cada parada até o destino
    df_temp["Distancia_destino"] = df_temp.apply(
        lambda p: haversine(
            destino_teste[0],
            destino_teste[1],
            p["py"],
            p["px"]
        ),
        axis=1
    )

    # Identificar as paradas mais próximas
    parada_origem = df_temp.loc[
        df_temp["Distancia_origem"].idxmin()
    ]

    parada_destino = df_temp.loc[
        df_temp["Distancia_destino"].idxmin()
    ]

    # Recuperar os dados da linha
    info = metadados_linhas[
        metadados_linhas["Codigo_linha"] == codigo
    ]

    if info.empty:
        continue

    info = info.iloc[0]

    resultados_paradas.append({
        "Codigo_linha": codigo,
        "Letreiro": info["Letreiro"],
        "Sentido": info["Sentido"],
        "Total_paradas": len(df_temp),

        "Embarque": parada_origem["np"],
        "Distancia_embarque": round(
            parada_origem["Distancia_origem"], 2
        ),

        "Desembarque": parada_destino["np"],
        "Distancia_desembarque": round(
            parada_destino["Distancia_destino"], 2
        ),

        "Origem_atendida": (
            parada_origem["Distancia_origem"] <= raio_paradas
        ),

        "Destino_atendido": (
            parada_destino["Distancia_destino"] <= raio_paradas
        )
    })

# Apresentar os resultados
df_analise_8000 = pd.DataFrame(resultados_paradas)

print("ANÁLISE DAS VARIANTES DA LINHA 8000")

display(df_analise_8000)

Erro na linha 2506: 401 Client Error: Unauthorized for url: https://api.olhovivo.sptrans.com.br/v2.1/Parada/BuscarParadasPorLinha?codigoLinha=2506
Erro na linha 35274: 401 Client Error: Unauthorized for url: https://api.olhovivo.sptrans.com.br/v2.1/Parada/BuscarParadasPorLinha?codigoLinha=35274
ANÁLISE DAS VARIANTES DA LINHA 8000


,Codigo_linha,Letreiro,Sentido,Total_paradas,Embarque,Distancia_embarque,Desembarque,Distancia_desembarque,Origem_atendida,Destino_atendido
0,1273,8000-10,1,14,ANGELICA B/C,0.00,PARADA PALMEIRAS B/C,0.00,True,True
1,34041,8000-10,2,13,ANGELICA C/B,20.89,PARADA PALMEIRAS C/B,29.53,True,True


In [20]:
from getpass import getpass

# Solicitar novamente o token
token_sptrans = getpass("Token SPTrans: ")

# Limpar os cookies antigos
sessao_api.cookies.clear()

# Realizar nova autenticação
resposta_login = sessao_api.post(
    f"{BASE_URL}/Login/Autenticar",
    params={"token": token_sptrans},
    timeout=20
)

resposta_login.raise_for_status()

autenticado = resposta_login.json()

if autenticado is True:
    print("Autenticação renovada com sucesso!")
else:
    print("Falha ao renovar a autenticação.")

# Remover a variável com o token
del token_sptrans

Autenticação renovada com sucesso!


In [21]:
codigo = 2506

resposta = sessao_api.get(
    f"{BASE_URL}/Parada/BuscarParadasPorLinha",
    params={"codigoLinha": codigo},
    timeout=30
)

resposta.raise_for_status()

paradas = resposta.json()

print("Quantidade de paradas:", len(paradas))

# Guardar os dados para reutilização
paradas_por_linha[codigo] = paradas

display(pd.DataFrame(paradas).head(10))

Quantidade de paradas: 14


,cp,np,ed,py,px
0,7014417,ANGELICA B/C,AV ANGELICA,-23.534564,-46.654302
1,60016784,PARADA PALMEIRAS B/C,R PADRE ANTONIO TOMAS/ AV POMPEIA,-23.525799,-46.679251
2,60016786,ANTARTICA B/C,PC SOUSA ARANHA/ VDTO ANTARTICA,-23.526523,-46.673588
3,60016788,BABY BARIONY B/C,R DONA GERMAINE BUCHARD/ R MINISTRO GODOI,-23.528131,-46.670652
4,60016790,PQ. DA ÁGUA BRANCA B/C,R MINISTRO GODOI/ R LINCOLN ALBUQUERQUE,-23.530210,-46.666900
5,480014612,SCIPIAO B/C,R SCIPIAO/ R AURELIA,-23.524940,-46.698995
6,480014613,VESPASIANO B/C,R VESPASIANO/ R CAIO GRACO,-23.524994,-46.695297
7,480014614,CORNELIA,R CLAUDIO/ R CRASSO,-23.525029,-46.690074
8,480014961,CATAO B/C,R FAUSTOLO/ R ROMA,-23.523407,-46.700541
9,610014603,SESC POMPEIA B/C,R TURIASSU/ R BARAO DO BANANAL,-23.525170,-46.683408


In [22]:
codigo = 35274

resposta = sessao_api.get(
    f"{BASE_URL}/Parada/BuscarParadasPorLinha",
    params={"codigoLinha": codigo},
    timeout=30
)

resposta.raise_for_status()

paradas = resposta.json()

print("Quantidade de paradas:", len(paradas))

paradas_por_linha[codigo] = paradas

display(pd.DataFrame(paradas).head(10))

Quantidade de paradas: 13


,cp,np,ed,py,px
0,614432,BABY BARIONY C/B,R PEDRO MACHADO,-23.527606,-46.671514
1,7014416,ANGELICA C/B,R DR. ALBUQUERQUE LINS,-23.534377,-46.654322
2,60014604,SESC POMPEIA C/B,VDTO POMPEIA/ R JOAQUIM FERREIRA,-23.525262,-46.682254
3,60016783,PARADA PALMEIRAS C/B,R PADRE ANTONIO TOMAS/ AV POMPEIA,-23.525653,-46.679493
4,60016785,ANTARTICA C/B,R ENGENHEIRO STEVENSON/ PC SOUSA ARANHA,-23.526220,-46.674187
5,60016789,PQ. DA ÁGUA BRANCA C/B,R DONA GERMAINE BUCHARD/ R MINISTRO GODOI,-23.529636,-46.667777
6,480014608,TIBERIO C/B,R TIBERIO/ R MENFIS,-23.522875,-46.688219
7,480014609,DUILIO C/B,R DUILIO/ R CLAUDIO,-23.522627,-46.692121
8,480014610,TENDAL DA LAPA C/B,R ESPARTACO/ R MORAIS E SILVA,-23.522339,-46.696246
9,700006085,ROSA E SILVA C/B,R OLIMPIA DE ALMEIDA PRADO/ R LOPES DE OLIVEIRA,-23.532629,-46.657465


In [24]:
codigos = [1273, 34041, 2506, 35274]

resultados = []

for codigo in codigos:

    df = pd.DataFrame(paradas_por_linha[codigo]).copy()

    df["py"] = pd.to_numeric(df["py"], errors="coerce")
    df["px"] = pd.to_numeric(df["px"], errors="coerce")

    df = df.dropna(subset=["py", "px"])

    # Distâncias até origem e destino
    df["dist_origem"] = df.apply(
        lambda p: haversine(
            origem_teste[0], origem_teste[1],
            p["py"], p["px"]
        ), axis=1
    )

    df["dist_destino"] = df.apply(
        lambda p: haversine(
            destino_teste[0], destino_teste[1],
            p["py"], p["px"]
        ), axis=1
    )

    embarque = df.loc[df["dist_origem"].idxmin()]
    desembarque = df.loc[df["dist_destino"].idxmin()]

    linha = df_linhas_monitoradas[
        df_linhas_monitoradas["Codigo_linha"] == codigo
    ].iloc[0]

    resultados.append({
        "Codigo": codigo,
        "Letreiro": linha["Letreiro"],
        "Sentido": linha["Sentido"],
        "Paradas": len(df),
        "Embarque": embarque["np"],
        "Distancia_origem": round(embarque["dist_origem"], 2),
        "Desembarque": desembarque["np"],
        "Distancia_destino": round(desembarque["dist_destino"], 2),
        "Origem_atendida": embarque["dist_origem"] <= 500,
        "Destino_atendido": desembarque["dist_destino"] <= 500
    })

df_analise_8000 = pd.DataFrame(resultados)

display(df_analise_8000)

comparacoes = [
    (1273, 2506, "Sentido 1"),
    (34041, 35274, "Sentido 2")
]

for codigo_a, codigo_b, sentido in comparacoes:

    paradas_a = set(
        pd.DataFrame(paradas_por_linha[codigo_a])["cp"]
    )

    paradas_b = set(
        pd.DataFrame(paradas_por_linha[codigo_b])["cp"]
    )

    print(f"\n{sentido}")
    print("Paradas em comum:", len(paradas_a & paradas_b))
    print("Paradas diferentes:", len(paradas_a ^ paradas_b))
    print("Listas idênticas:", paradas_a == paradas_b)

,Codigo,Letreiro,Sentido,Paradas,Embarque,Distancia_origem,Desembarque,Distancia_destino,Origem_atendida,Destino_atendido
0,1273,8000-10,1,14,ANGELICA B/C,0.00,PARADA PALMEIRAS B/C,0.00,True,True
1,34041,8000-10,2,13,ANGELICA C/B,20.89,PARADA PALMEIRAS C/B,29.53,True,True
2,2506,8000-1,1,14,ANGELICA B/C,0.00,PARADA PALMEIRAS B/C,0.00,True,True
3,35274,8000-1,2,13,ANGELICA C/B,20.89,PARADA PALMEIRAS C/B,29.53,True,True



Sentido 1
Paradas em comum: 14
Paradas diferentes: 0
Listas idênticas: True

Sentido 2
Paradas em comum: 13
Paradas diferentes: 0
Listas idênticas: True


In [25]:
pares = [
    (1273, 2506, "Sentido 1"),
    (34041, 35274, "Sentido 2")
]

for codigo_a, codigo_b, sentido in pares:

    lista_a = [
        p["cp"] for p in paradas_por_linha[codigo_a]
    ]

    lista_b = [
        p["cp"] for p in paradas_por_linha[codigo_b]
    ]

    print(f"\n{sentido}")
    print("Mesma quantidade:", len(lista_a) == len(lista_b))
    print("Mesma ordem:", lista_a == lista_b)


Sentido 1
Mesma quantidade: True
Mesma ordem: True

Sentido 2
Mesma quantidade: True
Mesma ordem: True


In [26]:
# Selecionar uma pequena amostra de linhas candidatas

amostra = candidatas.copy()

amostra["Pontuacao"] = (
    amostra["Veiculo_mais_proximo_origem"]
    + amostra["Veiculo_mais_proximo_destino"]
)

codigos_teste = list(
    amostra.nsmallest(8, "Pontuacao")["Codigo_linha"]
)

# Acrescentar duas linhas que já conhecemos
codigos_teste.extend([1273, 34041])

codigos_teste = list(dict.fromkeys(codigos_teste))

resultados = []

for codigo in codigos_teste:

    # Reutilizar dados já consultados
    if codigo in paradas_por_linha:
        dados = paradas_por_linha[codigo]

    else:
        resposta = sessao_api.get(
            f"{BASE_URL}/Parada/BuscarParadasPorLinha",
            params={"codigoLinha": int(codigo)},
            timeout=30
        )

        if resposta.status_code == 401:
            print("Sessão expirada. Linha:", codigo)
            break

        resposta.raise_for_status()

        dados = resposta.json()
        paradas_por_linha[codigo] = dados

    df = pd.DataFrame(dados)

    if df.empty:
        continue

    df["py"] = pd.to_numeric(df["py"], errors="coerce")
    df["px"] = pd.to_numeric(df["px"], errors="coerce")

    df = df.dropna(subset=["py", "px"])

    if df.empty:
        continue

    df["Distancia_origem"] = df.apply(
        lambda p: haversine(
            *origem_teste,
            p["py"],
            p["px"]
        ),
        axis=1
    )

    df["Distancia_destino"] = df.apply(
        lambda p: haversine(
            *destino_teste,
            p["py"],
            p["px"]
        ),
        axis=1
    )

    embarque = df.loc[df["Distancia_origem"].idxmin()]
    desembarque = df.loc[df["Distancia_destino"].idxmin()]

    info = metadados_linhas[
        metadados_linhas["Codigo_linha"] == codigo
    ]

    if info.empty:
        continue

    info = info.iloc[0]

    resultados.append({
        "Codigo": codigo,
        "Letreiro": info["Letreiro"],
        "Embarque": embarque["np"],
        "Distancia_embarque": round(
            embarque["Distancia_origem"], 1
        ),
        "Desembarque": desembarque["np"],
        "Distancia_desembarque": round(
            desembarque["Distancia_destino"], 1
        ),
        "Origem_atendida": embarque["Distancia_origem"] <= 500,
        "Destino_atendido": desembarque["Distancia_destino"] <= 500
    })

df_teste_candidatas = pd.DataFrame(resultados)

display(df_teste_candidatas)

,Codigo,Letreiro,Embarque,Distancia_embarque,Desembarque,Distancia_desembarque,Origem_atendida,Destino_atendido
0,33200,8594-10,ANGELICA C/B,20.9,PARADA PALMEIRAS C/B,29.5,True,True
1,34033,8400-10,ANGELICA C/B,20.9,PARADA PALMEIRAS C/B,29.5,True,True
2,438,8622-10,ANGELICA B/C,0.0,PARADA PALMEIRAS B/C,0.0,True,True
3,1339,8615-10,ANGELICA B/C,0.0,PARADA PALMEIRAS B/C,0.0,True,True
4,432,8594-10,ANGELICA B/C,0.0,PARADA PALMEIRAS B/C,0.0,True,True
5,575,8528-10,ANGELICA B/C,0.0,CETET - B/C,958.8,True,False
6,1273,8000-10,ANGELICA B/C,0.0,PARADA PALMEIRAS B/C,0.0,True,True
7,34041,8000-10,ANGELICA C/B,20.9,PARADA PALMEIRAS C/B,29.5,True,True


In [27]:
# Consultar previsões da linha 8622-10
codigo_linha_teste = 438

resposta = sessao_api.get(
    f"{BASE_URL}/Previsao/Linha",
    params={"codigoLinha": codigo_linha_teste},
    timeout=30
)

resposta.raise_for_status()

dados = resposta.json()

if isinstance(dados, dict):

    paradas = dados.get("ps") or []
    registros = []

    for parada in paradas:

        for veiculo in parada.get("vs") or []:

            registros.append({
                "Prefixo": veiculo.get("p"),
                "Parada": parada.get("np"),
                "Codigo_parada": parada.get("cp"),
                "Previsao": veiculo.get("t")
            })

    df_sequencia = pd.DataFrame(registros)

    print("Horário de referência:", dados.get("hr"))
    print("Paradas com previsão:", len(paradas))

    if not df_sequencia.empty:

        # Mostrar veículos que aparecem em mais de uma parada
        repetidos = df_sequencia[
            df_sequencia.duplicated(
                subset=["Prefixo"],
                keep=False
            )
        ]

        display(
            repetidos.sort_values(
                ["Prefixo", "Previsao"]
            )
        )

    else:

        print("Nenhuma previsão disponível no momento.")

else:

    print("A API não retornou os dados esperados.")

Horário de referência: 11:40
Paradas com previsão: 13


,Prefixo,Parada,Codigo_parada,Previsao
51,11027,SCIPIAO B/C,480014612,12:32
60,11027,VESPASIANO B/C,480014613,12:34
25,11027,CORNELIA,480014614,12:35
56,11027,SESC POMPEIA B/C,610014603,12:39
36,11027,PARADA PALMEIRAS B/C,60016784,12:41
...,...,...,...,...
42,11076,ROSA E SILVA C/B,700006086,11:41
6,11076,ANGELICA B/C,7014417,11:43
26,11076,NOTHMANN C/B,700016866,11:45
0,11076,ANA CINTRA B/C,700016623,11:46


In [28]:
# Identificar as duas paradas de interesse
origem_nome = "ANGELICA B/C"
destino_nome = "PARADA PALMEIRAS B/C"

# Filtrar previsões dessas paradas
previsoes_alvo = df_sequencia[
    df_sequencia["Parada"].isin([
        origem_nome,
        destino_nome
    ])
].copy()

print("PREVISÕES NOS PONTOS DE INTERESSE")

display(
    previsoes_alvo.sort_values(
        ["Prefixo", "Parada"]
    )
)

# Identificar veículos presentes nas duas paradas
veiculos_origem = set(
    previsoes_alvo.loc[
        previsoes_alvo["Parada"] == origem_nome,
        "Prefixo"
    ]
)

veiculos_destino = set(
    previsoes_alvo.loc[
        previsoes_alvo["Parada"] == destino_nome,
        "Prefixo"
    ]
)

veiculos_comuns = veiculos_origem.intersection(
    veiculos_destino
)

print("\nVeículos com previsão nas duas paradas:")
print(veiculos_comuns)

if veiculos_comuns:

    print("\nPREVISÕES DOS VEÍCULOS COMPARTILHADOS")

    display(
        previsoes_alvo[
            previsoes_alvo["Prefixo"].isin(veiculos_comuns)
        ].sort_values(["Prefixo", "Previsao"])
    )

else:

    print(
        "Nenhum veículo apresentou previsão "
        "nas duas paradas nesta consulta."
    )

PREVISÕES NOS PONTOS DE INTERESSE


,Prefixo,Parada,Codigo_parada,Previsao
11,11027,ANGELICA B/C,7014417,12:50
36,11027,PARADA PALMEIRAS B/C,60016784,12:41
9,11046,ANGELICA B/C,7014417,12:26
34,11046,PARADA PALMEIRAS B/C,60016784,12:17
7,11049,ANGELICA B/C,7014417,11:55
32,11049,PARADA PALMEIRAS B/C,60016784,11:46
10,11062,ANGELICA B/C,7014417,12:38
35,11062,PARADA PALMEIRAS B/C,60016784,12:29
8,11074,ANGELICA B/C,7014417,12:06
33,11074,PARADA PALMEIRAS B/C,60016784,11:57



Veículos com previsão nas duas paradas:
{'11046', '11049', '11074', '11027', '11062'}

PREVISÕES DOS VEÍCULOS COMPARTILHADOS


,Prefixo,Parada,Codigo_parada,Previsao
36,11027,PARADA PALMEIRAS B/C,60016784,12:41
11,11027,ANGELICA B/C,7014417,12:50
34,11046,PARADA PALMEIRAS B/C,60016784,12:17
9,11046,ANGELICA B/C,7014417,12:26
32,11049,PARADA PALMEIRAS B/C,60016784,11:46
7,11049,ANGELICA B/C,7014417,11:55
35,11062,PARADA PALMEIRAS B/C,60016784,12:29
10,11062,ANGELICA B/C,7014417,12:38
33,11074,PARADA PALMEIRAS B/C,60016784,11:57
8,11074,ANGELICA B/C,7014417,12:06


In [29]:
# Pesquisar as duas direções da linha 8622

resposta = sessao_api.get(
    f"{BASE_URL}/Linha/Buscar",
    params={"termosBusca": "8622"},
    timeout=30
)

resposta.raise_for_status()

dados_8622 = resposta.json()

df_8622 = pd.DataFrame(dados_8622)

print("Linhas encontradas:", len(df_8622))

display(df_8622)

Linhas encontradas: 2


,cl,lc,lt,sl,tl,tp,ts
0,438,False,8622,1,10,PÇA. RAMOS DE AZEVEDO,MORRO DOCE
1,33206,False,8622,2,10,PÇA. RAMOS DE AZEVEDO,MORRO DOCE


In [30]:
codigo = 33206

resposta = sessao_api.get(
    f"{BASE_URL}/Parada/BuscarParadasPorLinha",
    params={"codigoLinha": codigo},
    timeout=30
)

resposta.raise_for_status()

paradas = resposta.json()

paradas_por_linha[codigo] = paradas

df = pd.DataFrame(paradas)

# Calcular distância até a origem
df["Distancia_origem"] = df.apply(
    lambda p: haversine(
        *origem_teste, p["py"], p["px"]
    ),
    axis=1
)

# Calcular distância até o destino
df["Distancia_destino"] = df.apply(
    lambda p: haversine(
        *destino_teste, p["py"], p["px"]
    ),
    axis=1
)

embarque = df.loc[df["Distancia_origem"].idxmin()]
desembarque = df.loc[df["Distancia_destino"].idxmin()]

print("Total de paradas:", len(df))

print("\nPONTO DE EMBARQUE:")
display(embarque[["np", "cp", "Distancia_origem"]])

print("\nPONTO DE DESEMBARQUE:")
display(desembarque[["np", "cp", "Distancia_destino"]])

print("\nOrigem atendida:", embarque["Distancia_origem"] <= 500)
print("Destino atendido:", desembarque["Distancia_destino"] <= 500)

Total de paradas: 13

PONTO DE EMBARQUE:


np                  ANGELICA C/B
cp                       7014416
Distancia_origem       20.893176
Name: 1, dtype: object


PONTO DE DESEMBARQUE:


np                   PARADA PALMEIRAS C/B
cp                               60016783
Distancia_destino               29.534561
Name: 3, dtype: object


Origem atendida: True
Destino atendido: True


In [31]:
# Consultar previsões da linha 8622 - Sentido 2

codigo_linha = 33206

resposta = sessao_api.get(
    f"{BASE_URL}/Previsao/Linha",
    params={"codigoLinha": codigo_linha},
    timeout=30
)

resposta.raise_for_status()

dados = resposta.json()

registros = []

if isinstance(dados, dict):

    for parada in dados.get("ps") or []:

        for veiculo in parada.get("vs") or []:

            registros.append({
                "Prefixo": str(veiculo.get("p")),
                "Parada": parada.get("np"),
                "Codigo_parada": parada.get("cp"),
                "Previsao": veiculo.get("t")
            })

df_previsoes_33206 = pd.DataFrame(registros)

print("Horário de referência:", dados.get("hr"))
print("Previsões encontradas:", len(df_previsoes_33206))

display(df_previsoes_33206)

Horário de referência: 11:58
Previsões encontradas: 0


""


In [33]:
if df_previsoes_33206.empty:

    print("Não existem previsões disponíveis para esta linha.")
    print("Não foi possível verificar a sequência dos pontos.")

else:

    codigo_origem = 7014416
    codigo_destino = 60016783

    origem = df_previsoes_33206[
        df_previsoes_33206["Codigo_parada"] == codigo_origem
    ]

    destino = df_previsoes_33206[
        df_previsoes_33206["Codigo_parada"] == codigo_destino
    ]

    comparacao = origem.merge(
        destino,
        on="Prefixo",
        suffixes=("_origem", "_destino")
    )

    if comparacao.empty:

        print("Nenhum veículo possui previsão nas duas paradas.")

    else:

        print("Veículos encontrados nas duas paradas:")

        display(comparacao[
            [
                "Prefixo",
                "Previsao_origem",
                "Previsao_destino"
            ]
        ])

Não existem previsões disponíveis para esta linha.
Não foi possível verificar a sequência dos pontos.
